In [1]:
import sys, os, json, pickle
import pandas as pd
from pathlib import Path
import torch
from sklearn.metrics import r2_score

sys.path.append('../../../../src/fluprofiler')
sys.path.append('../../')

from experiment_tools import (
    find_repo_root, default_exp_id, make_run_dirs, generate_matrix,
    load_data_and_dataloaders, evaluate_step
)
from models.architectures import fluProfiler_v0_1, fluProfiler_Config

## inference

In [2]:
season = '2023NH'

# ---------- 1) 路径/数据 ----------
_CWD = Path.cwd().resolve()
_REPO_ROOT = find_repo_root(_CWD)     # notebook 下用 cwd 找 repo root
root_path = str(_REPO_ROOT) + "/"

data_path = root_path + "data/reverse_test/"
season_path = f"processed/test_{season}/"

exp_id = os.environ.get("FLUPROFILER_EXP_ID") or default_exp_id(_CWD, _REPO_ROOT)
tag = os.environ.get("FLUPROFILER_TAG") or "v3_0"
run_paths = make_run_dirs(_REPO_ROOT, exp_id=exp_id, tag=tag)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

data_loaders = load_data_and_dataloaders(
    data_path=data_path,
    season_path=season_path,
    batch_size=8,
    sample_limit=None,
    use_artificial=False,
    test_only=True
)
test_dataloader = data_loaders["test_dataloader"]
emb_dict = data_loaders["emb_dict"]

Loading tensor: 100%|██████████| 476/476 [00:18<00:00, 26.02file/s]


In [3]:
model_path = "/home/chenyh/workspace/fluProfiler/runs/reverse_tests/2023NH/v3_0/20260310_103444__v3_0_cached__pid2405894/checkpoints/2026-03-10_15-25-35.pth"
model = torch.load(model_path, map_location=device, weights_only=False)

model.eval()
with torch.no_grad():
    test_metrics = evaluate_step(model, test_dataloader, emb_dict, device, generate_matrix, return_predictions=True)

MAE: 0.87917
MSE: 1.27611
pearson correlation: 0.78801
spearman correlation: 0.74411
R2_score: 0.57030


## Analysis

In [13]:
test_data = pd.read_csv('/home/chenyh/workspace/fluProfiler/data/reverse_test/processed/test_2023NH/test.csv')
test_data['prediction'] = test_metrics['predictions']

In [14]:
print('H1N1 samples: ', len(test_data[test_data['Type'] == 'H1N1']))
print('H3N2 samples: ', len(test_data[test_data['Type'] == 'H3N2']))

H1N1 samples:  1096
H3N2 samples:  2406
